recuperation du dataset produit a la suite du nettoyage

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Définition propre du chemin vers le fichier nettoyé
BASE_DIR = Path.cwd().parent  # Remonte à la racine du projet
DATA_PATH = BASE_DIR / "data" / "processed" / "comptages_prepares.csv"

# chargement avec precision sur le séparateur (;)
df = pd.read_csv(DATA_PATH, sep=";")

print(f"Données nettoyées chargées avec succès : {df.shape[0]} lignes et {df.shape[1]} colonnes.")

Données nettoyées chargées avec succès : 148376 lignes et 7 colonnes.


1. Variables temporelles en heure de Paris

In [2]:
# Premierement Convertir en datetime et appliquer le fuseau horaire de Paris
df["t_1h"] = pd.to_datetime(df["t_1h"], utc=True).dt.tz_convert("Europe/Paris")

# Extraction des composantes temporelles dont on se servira pour la modélisation
df["heure"] = df["t_1h"].dt.hour
df["jour_semaine"] = df["t_1h"].dt.dayofweek + 1 # +1 pour revenir a la convention traditionnelle Lundi = 1 et Dimanche =7
df["mois"] = df["t_1h"].dt.month
df["nom_jour"] = df["t_1h"].dt.day_name()

# Affichage juste pour le controle
print("Vérification du type et du fuseau horaire")
print(f"Type de la colonne t_1h : {df['t_1h'].dtype}")
print(f"Exemple d'horodatage converti : {df['t_1h'].iloc[0]}")

print("\nAperçu des nouvelles variables temporelles créées")
print(df[["iu_ac", "libelle","t_1h", "nom_jour", "heure", "jour_semaine", "mois"]].head(5))



Vérification du type et du fuseau horaire
Type de la colonne t_1h : datetime64[us, Europe/Paris]
Exemple d'horodatage converti : 2026-09-01 01:00:00+02:00

Aperçu des nouvelles variables temporelles créées
   iu_ac      libelle                      t_1h nom_jour  heure  jour_semaine  \
0   5601  Bd_Grenelle 2026-09-01 01:00:00+02:00  Tuesday      1             2   
1   5602  Bd_Grenelle 2026-09-01 01:00:00+02:00  Tuesday      1             2   
2   5604  Bd_Grenelle 2026-09-01 01:00:00+02:00  Tuesday      1             2   
3   5606  Bd_Grenelle 2026-09-01 01:00:00+02:00  Tuesday      1             2   
4   5608  Bd_Grenelle 2026-09-01 01:00:00+02:00  Tuesday      1             2   

   mois  
0     9  
1     9  
2     9  
3     9  
4     9  


2. Encodage

In [3]:
# Une vérification par calcul du taux d'occupation (k) moyen par etat_trafic sur chaque axe
ordre_physique = (df.groupby(["libelle", "etat_trafic"])["k"].mean().unstack("etat_trafic")
    .round(2))
print("Taux d'occupation moyen (k) par état :")
print(ordre_physique)

# Vérification des modalités uniques existantes dans la colonne
print("\n Modalités uniques de etat_barre :", df["etat_barre"].unique())

Taux d'occupation moyen (k) par état :
etat_trafic  Bloqué  Fluide  Inconnu  Pré-saturé  Saturé
libelle                                                 
Av_Foch       59.75    5.61    11.08       20.30   37.01
Bd_Barbes     60.30    6.76    12.66       20.28   38.82
Bd_Grenelle   55.01    4.10     5.90       20.67   34.93

 Modalités uniques de etat_barre : <StringArray>
['Ouvert', 'Invalide', 'Barré']
Length: 3, dtype: str


« Les données montrent la hiérarchie  du taux d'occupation k : Fluide < Pré-saturé < Saturé < Bloqué. Il existe donc un ordre naturel évident. Nous optons pour n encodage ordinal ( 0, 1, 2, 3).

In [4]:
# Application de l'encodage sur etat_trafic
mapping_etat = {"Fluide": 0, "Pré-saturé": 1, "Saturé": 2, "Bloqué": 3}
df["etat_trafic_encoded"] = df["etat_trafic"].map(mapping_etat)

#Appercu
print(
    df[["etat_trafic", "etat_trafic_encoded"]]
    .drop_duplicates()
    .sort_values(by="etat_trafic_encoded")
)

# Application du mapping binaire (0/1)
mapping_barre = {"Ouvert": 0, "Barré": 1,"Invalide": np.nan,}
df["etat_barre_encoded"] = (
    df["etat_barre"].map(mapping_barre)
)

#Appercu
print("\n Mapping etat_barre")
print(
    df[["etat_barre", "etat_barre_encoded"]]
    .value_counts(dropna=False)
    .reset_index(name="effectif")
)

# Encodage One-Hot de la variable libelle car elle est nominale
df = pd.get_dummies(
    df, columns=["libelle"], prefix="axe", dtype=int, drop_first=False
)

#Appercu
cols_axes = [col for col in df.columns if col.startswith("axe_")]
print("\n Nouvelles colonnes d'axes créées")
print(df[cols_axes].head(15))



     etat_trafic  etat_trafic_encoded
0         Fluide                  0.0
100   Pré-saturé                  1.0
176       Saturé                  2.0
2774      Bloqué                  3.0
8        Inconnu                  NaN

 Mapping etat_barre
  etat_barre  etat_barre_encoded  effectif
0     Ouvert                 0.0    132134
1   Invalide                 NaN     15744
2      Barré                 1.0       498

 Nouvelles colonnes d'axes créées
    axe_Av_Foch  axe_Bd_Barbes  axe_Bd_Grenelle
0             0              0                1
1             0              0                1
2             0              0                1
3             0              0                1
4             0              0                1
5             1              0                0
6             1              0                0
7             1              0                0
8             1              0                0
9             1              0                0
10            1 

3. Valeurs aberrantes : Comparaison de l'IQR a la realite physique

In [5]:
# Reconstitution de la colonne 'libelle' à partir des colonnes One-Hot créées pour nous permettre de calcul les bornes par axe
cols_axes = [col for col in df.columns if col.startswith("axe_")]

# On retire le préfixe 'axe_' pour retrouver le nom propre de la rue
df["libelle"] = (
    df[cols_axes].idxmax(axis=1).str.replace("axe_", "", regex=False)
)

# Vérification
print("Axe reconstitué avec succès :")
print(df[["libelle"] + cols_axes].head(15))

Axe reconstitué avec succès :
        libelle  axe_Av_Foch  axe_Bd_Barbes  axe_Bd_Grenelle
0   Bd_Grenelle            0              0                1
1   Bd_Grenelle            0              0                1
2   Bd_Grenelle            0              0                1
3   Bd_Grenelle            0              0                1
4   Bd_Grenelle            0              0                1
5       Av_Foch            1              0                0
6       Av_Foch            1              0                0
7       Av_Foch            1              0                0
8       Av_Foch            1              0                0
9       Av_Foch            1              0                0
10      Av_Foch            1              0                0
11      Av_Foch            1              0                0
12    Bd_Barbes            0              1                0
13    Bd_Barbes            0              1                0
14    Bd_Barbes            0              1            

In [6]:
# 1. Calcul des bornes IQR par axe
stats_iqr = (
    df.groupby("libelle")
    .agg(
        Q1=("q",lambda x: x.quantile(0.25)),
        Q3=("q",lambda x: x.quantile(0.75)),
        Max_Observe=("q", "max"),
        k_Max=("k", "max"),
    )
    .reset_index()
)

stats_iqr["IQR"] = stats_iqr["Q3"] - stats_iqr["Q1"]
stats_iqr["Borne_Haute_IQR"] = stats_iqr["Q3"] + 1.5 * stats_iqr["IQR"]

print("COMPARAISON DES BORNES IQR PAR AXE")
print(stats_iqr[["libelle", "Q1", "Q3", "Borne_Haute_IQR", "Max_Observe","k_Max" ]])

COMPARAISON DES BORNES IQR PAR AXE
       libelle     Q1     Q3  Borne_Haute_IQR  Max_Observe     k_Max
0      Av_Foch  361.0  857.0           1601.0      1357.00  82.76278
1    Bd_Barbes  328.0  486.0            723.0      5421.95  88.04500
2  Bd_Grenelle  125.0  300.0            562.5       627.00  56.62944


Analyse : 
- Sur le Bd_Barbes, le débit maximal de 5421,95 véhicules/h est physiquement impossible sur une seule voie urbaine. 
- Sur le Bd_Grenelle, la borne haute IQR est calculée à 562,50 véhicules/h, alors que le débit réel monte à 627 véhicules/h. Cette valeur est un pic légitime d'heure de pointe que nous decidons de garder et non une valeur aberrante. 
- Sur Av_Foch, la borne haute IQR 1601 véhicules/h est trop laxiste, nous remarquons qu'elle est superieure au debit max 1357 véhicules/h mais on conserve la valeur de ce debit max car elle reste realiste?

Traitement : Nous appliquerons simplement un plafonnage a 3000 (q > 3000 véhicules/h ) pour remplacer l'unique valeur aberrante par NaN sur Barbès sans altérer les vraies heures de pointe sur Grenelle et Foch. Nous considerons 3000 véhicules/h comme une limite realiste pour une voie urbaine.

In [7]:
# Plafonnage a 3000 et remplacement des aberrations par NaN
df.loc[df["q"] > 3000, "q"] = np.nan

4. Statistiques par troncon

In [8]:
# Table récapitulative des statistiques par axe
stats_troncons = df.groupby("libelle")[["q", "k"]].agg(["mean", "std", "median", "max"]).reset_index()
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
print("Statistiques descriptives par tronçon")
print(stats_troncons)

Statistiques descriptives par tronçon
       libelle           q                                     k                              
                      mean         std median     max       mean        std   median       max
0      Av_Foch  624.713185  307.068434  692.0  1357.0   8.952350   8.756712  6.00111  82.76278
1    Bd_Barbes  403.305331  120.003441  431.0  1675.0  10.905194  11.712624  7.46056  88.04500
2  Bd_Grenelle  211.845200  103.970647  221.0   627.0   5.340668   5.761027  3.71667  56.62944


1. Le remplacement des valeurs supérieures à 3000 véhicules/h par des NaN a efficacement corrigé l'anomalie du Boulevard Barbès, on observe que le débit max nettoyé repasse à une valeur réaliste de 1675 véhicules/h.
2. L'Avenue Foch est clairement l'axe le plus fréquenté avec une moyenne de 624 véhicules/h, suivie du Boulevard Barbès avec 403,31 véhicules/h et du Boulevard Grenelle avec 211,85 véhicules/h
3. Le Boulevard Barbès présente le taux d'occupation moyen le plus élevé 10,91 % et atteint une saturation maximale extrême à 88,05%, ce qui nous indique que c'est l'axe le plus souvent de bouchons en comparaison a l'avenue Foch et au Boulevard Grenelle.

In [9]:
# Export du fichier transformé
df.to_csv("../data/processed/comptages_transformes.csv", index=False, sep=";")